# RFM + K-Means Bank Customer Segmentation – Solution

**Short name (GitHub):** `BankRFM`

Worked answers for `BankRFM_Practice_Skeleton.ipynb`. Numbers below are from **this extract** (18,310 postings → 3,250 CIF, $2.59M posted flow), a banking adaptation of CustSeg — not a Call Report and not a credit book.


## Inline cheat-sheet (keep this cell visible)

See also **`BankRFM_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Line item vs CIF | row in `bank_transactions.csv` = one posting; RFM row = one `CustomerID` |
| TotalSum | \(Q \times\) UnitAmount after `Quantity > 0` (and usually `UnitAmount > 0`) |
| Snapshot | \(\text{snapshot} = \max(\text{TxnDate}) + 1\text{ day}\) |
| Recency | \((\text{snapshot} - \max_i t_i).\text{days}\) — smaller is warmer |
| Frequency | `nunique(TxnNo)`, **not** product-line `count` |
| Monetary | \(\sum Q\cdot A\) over the window (posted flow, not balance) |
| Skew fix | `np.log1p` on R, F, M **before** scaling |
| Scale | \(x'=(x-\mu)/\sigma\) on the log frame; keep \(\mu,\sigma\) for new CIF |
| Inertia | \(J=\sum_i\|x_i-c_{\ell_i}\|^2\) on the **scaled** matrix |
| Elbow / sil. | plot \(J(k)\); this extract likes \(k=4\) (ops capacity + elbow) |
| Name bins | groupby Cluster → **median** R/F/M in original units |
| PCA | a slide of the 3-D scaled space; PC1 ≈ value, PC2 ≈ recency |

**Order:** clean → RFM → log1p → scale → choose \(k\) → fit → profile originals → PCA last.

**Not a credit decision.** Clustering groups similar RFM rows. A human still names the bins and owns the treatment.


## Desired outcome

![flowchart](bankrfm_flowchart.png)

1. Load `data/bank_transactions.csv`. Drop missing `CustomerID`. Keep `Quantity > 0` (and `UnitAmount > 0` for a strict posted-flow book).
2. `TotalSum = Quantity * UnitAmount`. Parse `TxnDate` with `%d.%m.%Y %H:%M`. Cast `CustomerID` to int.
3. `snapshot_date = max(TxnDate) + 1 day`. Aggregate Recency / Frequency / Monetary per CIF.
4. `log1p` the three columns, then z-score. Do not overwrite the original RFM table.
5. Elbow + silhouette for \(k=1\ldots10\). Fit K-Means at the chosen \(k\) (we use 4).
6. Write `rfm["Cluster"] = labels`. Profile **medians** in original units and name the bins.
7. PCA to 2-D is a picture, not the model. Alternates, more practice, then turn the simulation knobs.


## 0. Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.cluster import KMeans, MiniBatchKMeans, AgglomerativeClustering
    from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
    from sklearn.decomposition import PCA
    from sklearn.metrics import silhouette_score
    HAS_SK = True
except ImportError:
    HAS_SK = False
    print("sklearn not found — BankRFM.py NumPy fallbacks will run the clustering cells.")

import BankRFM as br

plt.rcParams["figure.figsize"] = (8, 4.5)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 20)
print("sklearn", HAS_SK, "| BankRFM helpers ready")


## 1. Why RFM then K-Means in a bank

Core postings are the wrong grain for a relationship segment. RFM collapses the book to one row per CIF. K-Means then needs a roughly spherical cloud — hence log1p on the commercial-whale tail and a z-score before `.fit`.


## 2. Load and clean

In [ ]:
raw = pd.read_csv("data/bank_transactions.csv")
print("raw shape", raw.shape)
print(raw.dtypes)
print("missing\n", raw.isna().sum())
print(raw[["Quantity", "UnitAmount"]].describe())
print("R-reversals", raw["TxnNo"].astype(str).str.startswith("R").sum())
print(raw["Region"].value_counts().head())


In [ ]:
sales = raw.dropna(subset=["CustomerID"]).copy()
sales = sales[sales["Quantity"] > 0]
sales = sales[sales["UnitAmount"] > 0]
sales["TotalSum"] = sales["Quantity"] * sales["UnitAmount"]
sales["TxnDate"] = pd.to_datetime(sales["TxnDate"], format="%d.%m.%Y %H:%M")
sales["CustomerID"] = sales["CustomerID"].astype(int)

print("clean shape", sales.shape)
print("negative qty left", int((sales["Quantity"] < 0).sum()))
print("missing CIF", int(sales["CustomerID"].isna().sum()))
print("posted flow", round(sales["TotalSum"].sum(), 2))
print("CIF", sales["CustomerID"].nunique(), "postings", sales["TxnNo"].nunique())
print("window", sales["TxnDate"].min(), "→", sales["TxnDate"].max())


## 3. Look at the book before you cluster

In [ ]:
by_reg = sales.groupby("Region")["TotalSum"].sum().sort_values(ascending=False)
print(by_reg.head(6).round(2))
print("NY Metro + Northeast share",
      round(by_reg.reindex(["New York Metro", "Northeast"]).sum() / by_reg.sum(), 3))

fig, ax = plt.subplots()
clipped = sales["TotalSum"].clip(upper=sales["TotalSum"].quantile(0.99))
ax.hist(clipped, bins=40, color="#4c78a8", edgecolor="white")
ax.set_title("Line-item TotalSum (99th pct clip) — not yet RFM Monetary")
ax.set_xlabel("$")
plt.show()


## 4. Build the RFM table

In [ ]:
snapshot = sales["TxnDate"].max() + pd.Timedelta(days=1)
print("snapshot", snapshot)

rfm = sales.groupby("CustomerID").agg(
    Recency=("TxnDate", lambda x: (snapshot - x.max()).days),
    Frequency=("TxnNo", "nunique"),
    Monetary=("TotalSum", "sum"),
).reset_index()

print(rfm.head())
print(rfm[["Recency", "Frequency", "Monetary"]].describe().round(2))
print("skew", rfm[["Recency", "Frequency", "Monetary"]].skew().round(2).to_dict())
print("n", len(rfm))


## 5. Log1p then scale

In [ ]:
rfm_log, rfm_scaled, mu, sd = br.log_scale(rfm)
print("log skew", rfm_log.skew().round(2).to_dict())
print("scaled mean", rfm_scaled.mean(axis=0))
print("scaled std ", rfm_scaled.std(axis=0))


## 6. Elbow and silhouette

In [ ]:
ks, inertias = br.elbow_inertias(rfm_scaled, range(1, 11), random_state=42, n_init=8)
sils = [np.nan]
for k in ks[1:]:
    _, lab = br.fit_kmeans(rfm_scaled, k=k)
    sils.append(br.silhouette(rfm_scaled, lab))

print("inertia", [round(float(v), 1) for v in inertias])
print("sil    ", [None if (isinstance(v, float) and np.isnan(v)) else round(float(v), 3) for v in sils])

fig, ax1 = plt.subplots()
ax1.plot(list(ks), inertias, "o-", color="#4c78a8", label="inertia")
ax1.axvline(4, color="#e15759", ls="--", label="k=4")
ax1.set_xlabel("k"); ax1.set_ylabel("inertia")
ax2 = ax1.twinx()
ax2.plot(list(ks), sils, "s--", color="#59a14f", label="silhouette")
ax2.set_ylabel("silhouette")
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="center right")
ax1.set_title("Elbow + silhouette on log-scaled bank RFM")
ax1.grid(True, alpha=0.3)
plt.show()


## 7. Fit K-Means and attach labels

In [ ]:
k_optimal = 4
model, labels = br.fit_kmeans(rfm_scaled, k=k_optimal)
rfm["Cluster"] = labels
print(rfm["Cluster"].value_counts().sort_index().to_dict())


## 8. Profile in original units and name the bins

In [ ]:
cluster_summary = br.profile_clusters(rfm, rfm["Cluster"])
print(cluster_summary)

share_n = rfm.groupby("Cluster").size() / len(rfm)
share_$ = rfm.groupby("Cluster")["Monetary"].sum() / rfm["Monetary"].sum()
print("% CIF\n", (100 * share_n).round(1))
print("% flow\n", (100 * share_$).round(1))

NAME = br.name_from_medians(rfm, rfm["Cluster"])
print("names", NAME)
rfm["Segment"] = rfm["Cluster"].map(NAME)
print(rfm.groupby("Segment")[["Recency", "Frequency", "Monetary"]].median().round(1))


## 9. PCA is a slide, not the model

In [ ]:
Z, ev, comps = br.pca2(rfm_scaled)
# flip PC1 if Frequency loads negative so "value" reads left-to-right
if comps[0, 1] < 0:
    Z = Z.copy(); Z[:, 0] *= -1
    comps = comps.copy(); comps[0] *= -1

pca_df = pd.DataFrame(Z, columns=["PC1", "PC2"])
pca_df["Cluster"] = rfm["Cluster"].to_numpy()
pca_df["Segment"] = rfm["Segment"].to_numpy()

print("explained variance", np.round(ev, 3), "sum", round(float(np.sum(ev)), 3))
loadings = pd.DataFrame(np.asarray(comps).T, index=["Recency", "Frequency", "Monetary"],
                        columns=["PC1", "PC2"]).round(3)
print(loadings)

fig, ax = plt.subplots(figsize=(8, 6))
for seg, g in pca_df.groupby("Segment"):
    ax.scatter(g["PC1"], g["PC2"], s=16, alpha=0.65, label=seg)
ax.set_xlabel("PC1  relationship value")
ax.set_ylabel("PC2  recency contrast")
ax.set_title("CIF segments via PCA (scaled RFM)")
ax.legend(); ax.grid(True, alpha=0.25)
plt.show()


## 10. Alternate code that reaches the same idea

In [ ]:
# A. NumPy k-means vs sklearn (if present)
_, lab_np, J = br.kmeans_fit(rfm_scaled, k=4, seed=42)
print("A NumPy inertia", round(J, 1), "sizes", pd.Series(lab_np).value_counts().sort_index().to_dict())

# B. Robust-style scale: median/IQR after the same log1p
med = rfm_log.median().to_numpy()
iqr = (rfm_log.quantile(0.75) - rfm_log.quantile(0.25)).to_numpy()
iqr = np.where(iqr == 0, 1.0, iqr)
X_rob = (rfm_log.to_numpy() - med) / iqr
_, lab_rob = br.fit_kmeans(X_rob, k=4)
print("B robust-scale sizes", pd.Series(lab_rob).value_counts().sort_index().to_dict())

# C. quintile RFM card (R inverted)
rfm_q = br.rfm_quintile_scores(rfm)
print("C RFM score head")
print(rfm_q[["CustomerID", "R", "F", "M", "RFM"]].head())
print("C core-like (R>=4 & F>=4 & M>=4)", int(((rfm_q.R >= 4) & (rfm_q.F >= 4) & (rfm_q.M >= 4)).sum()))

# D. agglomerative if sklearn
if HAS_SK:
    lab_agg = AgglomerativeClustering(n_clusters=4).fit_predict(rfm_scaled)
    print("D Agglomerative sizes", pd.Series(lab_agg).value_counts().sort_index().to_dict())
else:
    print("D skipped (no sklearn)")

# E. SVD already used inside br.pca2 fallback
print("E PC variance from br.pca2", np.round(ev, 3))


## 11. More practice

In [ ]:
# P1. New York Metro only
ny = sales[sales["Region"] == "New York Metro"]
rfm_ny = br.build_rfm(ny, snapshot)
_, X_ny, _, _ = br.log_scale(rfm_ny)
_, lab_ny = br.fit_kmeans(X_ny, k=4)
print("P1 NY Metro CIF", len(rfm_ny))
print(br.profile_clusters(rfm_ny, lab_ny))

# P2. trailing 180 days
cut = snapshot - pd.Timedelta(days=180)
recent = sales[sales["TxnDate"] >= cut]
rfm_180 = br.build_rfm(recent, snapshot)
print("P2 180d CIF", len(rfm_180), "flow", round(rfm_180["Monetary"].sum(), 2))

# P3. one new CIF scored with the training μ,σ
new = np.log1p(np.array([[7.0, 10.0, 2500.0]]))
new_scaled = (new - mu) / sd
if model is not None:
    print("P3 predicted cluster", int(model.predict(new_scaled)[0]))
else:
    C, _, _ = br.kmeans_fit(rfm_scaled, k=4, seed=42)
    print("P3 predicted cluster", int(((new_scaled - C)**2).sum(1).argmin()))

# P4. drop Frequency
X_rm = rfm_scaled[:, [0, 2]]
_, lab_rm = br.fit_kmeans(X_rm, k=4)
print("P4 Recency+Monetary sizes", pd.Series(lab_rm).value_counts().sort_index().to_dict())


## 12. Simulation — turn the knobs

In [ ]:
K = 4
N = None
NOISE = 0.0
N_INIT = 10
SEED = 42

print(br.simulate(rfm_scaled, k=K, n=N, noise=NOISE, n_init=N_INIT, seed=SEED))

rows = []
for k in range(2, 9):
    for noise in (0.0, 0.15, 0.30):
        rows.append(br.simulate(rfm_scaled, k=k, n=N, noise=noise, n_init=8, seed=SEED))
sim = pd.DataFrame([r.__dict__ for r in rows])
print(sim.pivot(index="k", columns="noise", values="silhouette").round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for noise, g in sim.groupby("noise"):
    axes[0].plot(g["k"], g["inertia"], "o-", label=f"noise={noise}")
    axes[1].plot(g["k"], g["silhouette"], "s-", label=f"noise={noise}")
axes[0].set_title("Inertia vs k"); axes[1].set_title("Silhouette vs k")
for ax in axes:
    ax.set_xlabel("k"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## Audience rewrite (Jočys checklist + McMurrey types)

| Audience | What they need | One sentence they should hear |
|----------|----------------|-------------------------------|
| Expert (retail-bank scientist) | inertia, silhouette, loadings, log+scale order | k=4 matches the elbow; PC1 (~80%) is value, PC2 (~18%) is recency; name bins from medians. |
| Technician (CRM / branch ops) | a four-row treatment table and the CIF keys | Relationship core = cluster 1 on this seed; suppress expensive win-back to cluster 0 until a human reviews. |
| Executive (Head of Retail / CFO) | concentration, not eigenvalues | 17% of CIF produce 87% of observed posted flow; the slipping mid-book is the cheapest win-back. |
| Nonspecialist | a branch-floor analogy | We sorted customers by *how lately, how often, how much they posted* — not by FICO or first name. |


## What this model can and cannot do

**Can** — group RFM rows, show flow concentration, give ops four named bins.

**Cannot** — forecast default or next product, mint a private-bank tier, travel to another charter without refitting, treat a cluster id as a risk rating.

**Top uses:** CRM design, retain-vs-nurture, whale-vs-mass separation, board concentration slides.
**Anti-uses:** automated fee waivers, scoring with a different scaler, clustering raw unlogged Monetary, using the bin as PD.


## Next steps

- Add product-mix shares as scaled columns.
- Try k=3 if Retail will only fund three treatments.
- Cap Monetary at the 99th percentile before log1p.
- Read `BankRFM_Strategy_Guide.docx` before cloning onto another core extract.
